In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "analysis":
    os.chdir(os.path.dirname(os.getcwd()))
    sys.path.append(os.getcwd())

full_runs_dir = "../logs_cluster/logs/full_runs/"


In [4]:
from pathlib import Path
import re
import pandas as pd

experiment_dirs = [
    "march/2026_03_11_kodak_2x2_big_synthesis",
    "january/2026_01_23_YCoCg_big_synthesis_Kodak",
]

required_text_sections = {
    "processing_image": "Processing image",
    "color_space": "Using color space",
    "image_arm": "Using image ARM",
    "encoder_gain": "Using encoder gain",
    "multi_region_arm": "Using multi-region image ARM",
}

quantized_pattern = re.compile(
    r"Final results after quantization: Loss: ([\d.eE+-]+), "
    r"Rate NN: ([\d.eE+-]+), Rate Latent: ([\d.eE+-]+), Rate Img: ([\d.eE+-]+)"
)

kodim_pattern = re.compile(r"kodim(\d{2})")

rows = []

for exp_dir in experiment_dirs:
    results_dir = Path(full_runs_dir) / exp_dir / "results"

    if not results_dir.exists():
        print(f"[MISSING FOLDER] {results_dir}")
        continue

    log_files = sorted(results_dir.glob("*.log"))
    if not log_files:
        print(f"[NO LOG FILES] {results_dir}")
        continue

    for log_path in log_files:
        text = log_path.read_text(encoding="utf-8", errors="replace")

        missing_sections = [
            name for name, marker in required_text_sections.items() if marker not in text
        ]

        quantized_match = quantized_pattern.search(text)
        if quantized_match is None:
            missing_sections.append("final_quantized_metrics")

        kodim_match = kodim_pattern.search(log_path.name)
        if kodim_match is None:
            missing_sections.append("kodim_two_digit_id")

        if missing_sections:
            print(
                f"[SKIP] {log_path.name} | missing: {', '.join(missing_sections)}"
            )
            continue

        loss, rate_nn, rate_latent, rate_img = map(float, quantized_match.groups())
        kodim_id = int(kodim_match.group(1))

        rows.append(
            {
                "experiment": exp_dir,
                "kodim_id": kodim_id,
                "log_file": log_path.name,
                "loss": loss,
                "rate_nn": rate_nn,
                "rate_latent": rate_latent,
                "rate_img": rate_img,
            }
        )

if not rows:
    print("No valid log files were parsed.")
else:
    df = pd.DataFrame(rows).sort_values(["experiment", "kodim_id"]).reset_index(drop=True)

    # Space-separated table output (easy to copy).
    print(df.to_string(index=False, float_format=lambda x: f"{x:.9f}"))


                                  experiment  kodim_id                                                log_file        loss     rate_nn  rate_latent    rate_img
january/2026_01_23_YCoCg_big_synthesis_Kodak         1        2026_01_28__17_32_04__coolchic_kodak_kodim01.log 3.660256863 0.020744324  0.184149861 3.455362797
january/2026_01_23_YCoCg_big_synthesis_Kodak         2        2026_01_28__17_31_05__coolchic_kodak_kodim02.log 3.062562704 0.020960490  0.042439781 2.999162436
january/2026_01_23_YCoCg_big_synthesis_Kodak         3        2026_01_28__17_59_55__coolchic_kodak_kodim03.log 2.973594427 0.022528754  0.154188380 2.796877384
january/2026_01_23_YCoCg_big_synthesis_Kodak         4        2026_01_28__18_02_36__coolchic_kodak_kodim04.log 3.114288092 0.020918952  0.062472876 3.030896425
january/2026_01_23_YCoCg_big_synthesis_Kodak         5        2026_01_29__00_18_23__coolchic_kodak_kodim05.log 3.805474520 0.021920098  0.131646067 3.651908159
january/2026_01_23_YCoCg_big_synthesis_K